# MMAction: phát hiện ngã sớm và kiểm tra chuyển trạng thái

Notebook này warm-start từ checkpoint `fall_wrestle_pilot_01`, giữ nguyên trọng số AVA và bộ phát hiện người. Chọn **Colab GPU** rồi chạy các cell theo thứ tự. Dữ liệu công khai cho phép thử cải thiện ngã ngay. Gói review chia sẻ hiện có một video vật thể thao giữ riêng để test; hai video còn lại chưa có mốc nhãn. Để dạy model ngừng báo đỏ khi xô xát kết thúc, gắn nhãn các đoạn `after_fight` trong gói review và đưa ZIP vào `MyDrive/SENTINEL/artifacts/action_review_v1/action_review_v1.zip`, sau đó đổi `RUN_TAG`.

Checkpoint mới chỉ là pilot. Báo cáo so sánh checkpoint cha và mới trên test giữ riêng; không tự cài vào ứng dụng. Cache đặc trưng và trạng thái train lưu trên Drive để chạy tiếp. Không dùng video vật thể thao làm nhãn xô xát dương.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, urllib.request, zipfile
from google.colab import drive

RUN_TAG = 'action_transition_v2_01'
EPOCHS = 40
TRAIN_MINUTES = 90
assert RUN_TAG and all(c.isalnum() or c in '_-' for c in RUN_TAG)
subprocess.run(['nvidia-smi'], check=True)
drive.mount('/content/drive', timeout_ms=180000)
DRIVE = Path('/content/drive/MyDrive/SENTINEL')
RUN = DRIVE / 'artifacts/mmaction_action_transition_v2' / RUN_TAG
RUN.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/action_transition_v2'); WORK.mkdir(exist_ok=True)
CODE = WORK / 'code'; CODE.mkdir(exist_ok=True)
DEMO = CODE / 'mmaction2_demo'
WEIGHTS = WORK / 'weights'; WEIGHTS.mkdir(exist_ok=True)
CACHE = RUN / 'feature_cache'
print('Output on Drive:', RUN)


In [ ]:
# Immutable project source; reruns use the same trainer and preprocessing.
SOURCE_REVISION = '20b19ea1dfb4a5e96600450cfc41300f6aaa908b'
source_zip = WORK / 'source.zip'
revision_marker = WORK / 'source_revision.txt'
if (DEMO/'train_action_transition_v2.py').is_file() and (
        not revision_marker.is_file() or revision_marker.read_text().strip() != SOURCE_REVISION):
    raise RuntimeError('Colab has code from another revision; restart runtime before rerunning')
if not (DEMO / 'train_action_transition_v2.py').is_file():
    url = 'https://codeload.github.com/TranDuon/anomaly-behavior-detector-system/zip/' + SOURCE_REVISION
    with urllib.request.urlopen(url, timeout=180) as response, source_zip.open('wb') as target:
        shutil.copyfileobj(response, target)
    with zipfile.ZipFile(source_zip) as package:
        for item in package.infolist():
            target = (WORK / item.filename).resolve()
            assert target.is_relative_to(WORK.resolve()), item.filename
        package.extractall(WORK)
    source = WORK / ('anomaly-behavior-detector-system-' + SOURCE_REVISION)
    shutil.copytree(source, CODE, dirs_exist_ok=True)
    revision_marker.write_text(SOURCE_REVISION)
manifest_path = CODE / 'benchmarks/gmdcsa24_v2_1_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['publisher_tree_sha'] == '5abac7693229900cf80f722e878fbb119211fc1c'
print('Pinned source:', SOURCE_REVISION)


In [ ]:
# Shared with me: SENTINEL/artifacts/mmaction_interaction_v1
SHARED_V1_FOLDER_ID = '1A5ymX6nzVErvFBtOJUqFWXDRFd220Kmi'
SHARED_WEIGHTS_FOLDER_ID = '1dFKaJPymXlyHEvIzjovxxzeSvqueDWzy'

def shared_drive_file(folder_id, name, destination):
    """Read a file shared with the Colab Google account; no manual upload."""
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
    query = f"'{folder_id}' in parents and name = '{name}' and trashed = false"
    found = service.files().list(q=query, fields='files(id,name,size)', page_size=100).execute()['files']
    if len(found) != 1:
        raise FileNotFoundError(f'{name} not found in shared Drive folder {folder_id}; select the Google account that can open the folder')
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.partial')
    with temporary.open('wb') as output:
        downloader = MediaIoBaseDownload(output, service.files().get_media(fileId=found[0]['id']), chunksize=8*1024*1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    temporary.replace(destination)
    print('Downloaded from shared Drive:', name, destination.stat().st_size, 'bytes')
    return destination


PARENT_SHA = 'ddb09180284c4093189c53b7decbfcc5b51fd2d9b7700536d104524bceb5c344'
PARENT_FOLDER_ID = '1CPnlYVWz_xJLWqyQbM6oULojAo0vYTDw'
parent_zip = DRIVE / 'artifacts/mmaction_interaction_fall_wrestle/fall_wrestle_pilot_01/fall_wrestle_parent_verified.zip'
if not parent_zip.is_file():
    parent_zip = shared_drive_file(PARENT_FOLDER_ID, 'fall_wrestle_parent_verified.zip',
                                   WORK/'fall_wrestle_parent_verified.zip')
with zipfile.ZipFile(parent_zip) as package:
    assert {'interaction.pt', 'interaction.json'} <= set(package.namelist())
    metadata = json.loads(package.read('interaction.json'))
    payload = package.read('interaction.pt')
    assert hashlib.sha256(payload).hexdigest() == metadata['checkpoint_sha256'] == PARENT_SHA
    (WEIGHTS/'interaction_fall_wrestle.pt').write_bytes(payload)
    (WEIGHTS/'interaction_fall_wrestle.json').write_bytes(package.read('interaction.json'))
print('Verified parent:', PARENT_SHA)


In [ ]:
# Tải trên Colab. Cache ở /content; không cần tải trước trên máy Windows.
fall_zip = WORK / 'gmdcsa24_v2.1.zip'
fall_url = 'https://zenodo.org/api/records/13354453/files/ekramalam/GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos-v2.1.zip/content'
subprocess.run(['curl', '-L', '--fail', '--retry', '5', '--retry-delay', '5', '-C', '-', '-o', str(fall_zip), fall_url], check=True)
assert fall_zip.stat().st_size == 1107545615, 'GMDCSA24 ZIP size mismatch'
md5 = hashlib.md5()
with fall_zip.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024*1024), b''):
        md5.update(chunk)
assert md5.hexdigest() == manifest['archive_md5'] == '3d36f2c5c1a666b99639e4e9fd843efb'
FALL_EXTRACT = WORK / 'fall'; FALL_EXTRACT.mkdir(exist_ok=True)
with zipfile.ZipFile(fall_zip) as package:
    assert sum(item.file_size for item in package.infolist()) < 5_000_000_000
    for item in package.infolist():
        assert (FALL_EXTRACT / item.filename).resolve().is_relative_to(FALL_EXTRACT.resolve()), item.filename
        assert ((item.external_attr >> 16) & 0o170000) != 0o120000, 'Symlink in archive'
    package.extractall(FALL_EXTRACT)
first = list(FALL_EXTRACT.rglob('Subject 1/Fall/01.mp4'))
assert len(first) == 1
FALL_ROOT = first[0].parents[2]
assert (FALL_ROOT / 'Subject 4/ADL/01.mp4').is_file()

SCFD_REVISION = 'ff83b7c521e5ca4eb67a212cc42dbc54f39d6290'
SCFD_ROOT = WORK / SCFD_REVISION
if not SCFD_ROOT.is_dir():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/seymanurakti/fight-detection-surv-dataset.git', str(SCFD_ROOT)], check=True)
    subprocess.run(['git', '-C', str(SCFD_ROOT), 'checkout', '--quiet', SCFD_REVISION], check=True)
assert subprocess.check_output(['git','-C',str(SCFD_ROOT),'rev-parse','HEAD'],text=True).strip() == SCFD_REVISION
assert len(list((SCFD_ROOT/'fight').glob('*.mp4'))) == 150
assert len(list((SCFD_ROOT/'noFight').glob('*.mp4'))) == 150
print('Data ready on Colab:', FALL_ROOT, SCFD_ROOT)


In [ ]:
# Môi trường riêng tương thích MMAction2 v1.2.0 và checkpoint v1.
os.environ.update(MPLBACKEND='Agg', CUDA_MODULE_LOADING='LAZY', OMP_NUM_THREADS='4')
ENV = WORK / 'env'; PY = ENV / 'bin/python'
def run(command, label, timeout=1800):
    log = RUN / (label + '.log')
    with log.open('a', encoding='utf-8') as output:
        result = subprocess.run([str(x) for x in command], stdout=output, stderr=subprocess.STDOUT, timeout=timeout)
    if result.returncode:
        raise RuntimeError(f'{label} failed; see {log}\n{log.read_text(encoding="utf-8")[-4000:]}')
    print(label, 'OK')
if not PY.is_file():
    run([sys.executable,'-m','pip','install','-q','uv==0.8.22'], 'install_uv')
    run(['uv','python','install','3.10'], 'install_python')
    run(['uv','venv','--python','3.10','--seed',ENV], 'create_env')
run([PY,'-m','pip','install','-q','numpy==1.26.4','setuptools==69.5.1','wheel'], 'install_base')
run([PY,'-m','pip','install','-q','torch==2.0.1+cu118','torchvision==0.15.2+cu118',
     '--index-url','https://download.pytorch.org/whl/cu118'], 'install_torch')
run([PY,'-m','pip','install','-q','mmcv==2.1.0','--only-binary=mmcv',
     '-f','https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html',
     'mmengine==0.10.7','mmdet==3.2.0','decord==0.6.0','einops==0.8.1',
     'importlib-metadata','scikit-learn==1.3.2','opencv-python==4.8.1.78'], 'install_openmmlab')
if not (DEMO / 'repo/.git').is_dir():
    run(['git','clone','--depth','1','--branch','v1.2.0','https://github.com/open-mmlab/mmaction2.git',DEMO/'repo'], 'clone_mmaction')
run([PY,'-m','pip','install','--no-deps','--no-build-isolation','-e',DEMO/'repo'], 'install_mmaction')
os.environ['PYTHONPATH'] = os.pathsep.join(map(str,(CODE,DEMO,DEMO/'repo')))
run([PY,'-c','import torch, mmcv; from mmcv.ops import roi_align; assert torch.cuda.is_available()'], 'check_cuda')


In [ ]:
# Tải trọng số gốc AVA + detector; dùng cache trên Drive nếu đã có.
WEIGHT_CACHE = DRIVE / 'artifacts/mmaction_interaction_fall_wrestle/public_weights'
WEIGHT_CACHE.mkdir(parents=True, exist_ok=True)
checks = {
 'action_slowonly_r101.pth': (
  'https://download.openmmlab.com/mmaction/detection/ava/slowonly_omnisource_pretrained_r101_8x8x1_20e_ava_rgb/slowonly_omnisource_pretrained_r101_8x8x1_20e_ava_rgb_20201217-16378594.pth',
  '163785949c5178484012a6593d12d6fcf4d36be404fa42b5899b314e91f4703b'),
 'person_faster_rcnn_r50.pth': (
  'https://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth',
  'a5d8aa15c75b961b9c73f28a6e92af370835639a2939ff890852b052a5256a5b')}
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
for name,(url,expected) in checks.items():
    target = WEIGHTS/name; cached = WEIGHT_CACHE/name
    if not cached.is_file() or sha256(cached)!=expected:
        legacy_candidates = [
            DRIVE/'artifacts/mmaction_interaction_v1/public_weights'/name,
            DRIVE/'artifacts/mmaction_indoor_review_v2/public_weights'/name,
        ]
        legacy = next((p for p in legacy_candidates if p.is_file() and sha256(p)==expected), None)
        if legacy is not None:
            shutil.copyfile(legacy,cached)
        else:
            temporary=cached.with_suffix('.partial')
            verified=False
            for attempt in range(1,3):
                temporary.unlink(missing_ok=True)
                request=urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0','Accept-Encoding':'identity'})
                with urllib.request.urlopen(request,timeout=180) as response,temporary.open('wb') as f:
                    shutil.copyfileobj(response,f)
                actual=sha256(temporary)
                if actual==expected:
                    temporary.replace(cached)
                    verified=True
                    break
                print(f'{name}: download {attempt}/2 got {temporary.stat().st_size} bytes, SHA-256 {actual}; expected {expected}')
                temporary.unlink(missing_ok=True)
            if not verified:
                shared_drive_file(SHARED_WEIGHTS_FOLDER_ID, name, cached)
                actual=sha256(cached)
                if actual!=expected:
                    raise ValueError(f'Shared Drive {name} has SHA-256 {actual}, expected {expected}')
    shutil.copyfile(cached,target)
print('AVA and person detector weights verified')


In [ ]:
# Prefer your reviewed ZIP in MyDrive; otherwise use the shared test-only pack.
review_zip = DRIVE / 'artifacts/action_review_v1/action_review_v1.zip'
review_root = WORK / 'review'
review_args = []
if not review_zip.is_file():
    review_zip = shared_drive_file('1RDprs7P9ag0Hm_ZMPmIdfbTP87OldfEk',
                                   'action_review_v1.zip', WORK/'action_review_v1.zip')
if review_zip.is_file():
    review_root.mkdir(exist_ok=True)
    with zipfile.ZipFile(review_zip) as package:
        for item in package.infolist():
            assert (review_root/item.filename).resolve().is_relative_to(review_root.resolve())
        package.extractall(review_root)
    review_manifest = review_root/'review.json'
    assert review_manifest.is_file()
    review_args = ['--review-manifest', review_manifest, '--review-root', review_root]
    print('Reviewed intervals loaded:', review_manifest)
else:
    print('No reviewed pack: public early-fall experiment only; no post-fight training claims.')


In [ ]:
base = [PY, DEMO/'train_action_transition_v2.py', '--manifest', manifest_path,
        '--fall-root', FALL_ROOT, '--scfd-root', SCFD_ROOT,
        '--weights', WEIGHTS, '--cache', CACHE, '--output', RUN,
        '--device', 'cuda:0', '--epochs', EPOCHS, '--minutes', TRAIN_MINUTES] + review_args
run(base + ['--stage', 'features'], 'extract_features', timeout=8*3600)
run(base + ['--stage', 'train'], 'fine_tune', timeout=(TRAIN_MINUTES+10)*60)


In [ ]:
report_path = RUN/'report.json'
assert report_path.is_file(), 'Training did not finish; inspect logs on Drive'
result = json.loads(report_path.read_text(encoding='utf-8'))
assert result['status'] == 'completed'
package = RUN/'mmaction_action_transition_v2_pilot.zip'
assert package.is_file()
print('PACKAGE:', package)
print('REPORT:', report_path)
print('Parent vs new test with detected people:', json.dumps(result['test_with_person'], indent=2))
print('Reviewed test actions:', json.dumps(result['test_reviewed_by_action'], indent=2))
print('This is a pilot. Install only after checking held-out temporal metrics.')
